# 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(
    style="darkgrid",
    palette="pastel",
    font="sans-serif",
    rc={"grid.linestyle": "--", "axes.edgecolor": "0.8"}
)

from collections import Counter
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 5
import re
import string
import nltk
import html
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import torch
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

seed = 5

C:\Users\zaida\anaconda3\envs\machine_learning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Dataset

In [2]:
test = pd.read_csv('test.csv')
train = pd.read_csv('train.csv')

In [3]:
# Train DataFrame
train

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [4]:
# Test DataFrame
test

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan
...,...,...,...,...
3258,10861,NaN,NaN,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...
3259,10865,NaN,NaN,Storm in RI worse than last hurricane. My city...
3260,10868,NaN,NaN,Green Line derailment in Chicago http://t.co/U...
3261,10874,NaN,NaN,MEG issues Hazardous Weather Outlook (HWO) htt...


# 3. EDA

## General Analysis

In [5]:
# Overlook
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        7613 non-null   int64
 1   keyword   7552 non-null   str  
 2   location  5080 non-null   str  
 3   text      7613 non-null   str  
 4   target    7613 non-null   int64
dtypes: int64(2), str(3)
memory usage: 1.2 MB


In [6]:
train.describe(include='str')

,keyword,location,text
count,7552,5080,7613
unique,221,3341,7503
top,fatalities,USA,11-Year-Old Boy Charged With Manslaughter of T...
freq,45,104,10


In [7]:
# Duplicate values
train.drop(columns='id').duplicated().sum()

np.int64(52)

In [8]:
# Drop duplicates
train = train.drop_duplicates(subset=[c for c in train.columns if c != 'id'])

In [9]:
# Duplicate text with contradictory labels
contradictory_labels = train.drop(columns=['id', 'target']).duplicated(keep=False)
train[contradictory_labels]

,id,keyword,location,text,target
610,881,bioterrorism,NaN,To fight bioterrorism sir.,1
624,898,bioterrorism,NaN,To fight bioterrorism sir.,0
2830,4068,displaced,Pedophile hunting ground,.POTUS #StrategicPatience is a strategy for #G...,1
2832,4076,displaced,Pedophile hunting ground,.POTUS #StrategicPatience is a strategy for #G...,0
3985,5662,floods,NaN,Who is bringing the tornadoes and floods. Who ...,1
4013,5699,floods,NaN,Who is bringing the tornadoes and floods. Who ...,0
4232,6012,hazardous,NaN,Caution: breathing may be hazardous to your he...,1
4235,6017,hazardous,NaN,Caution: breathing may be hazardous to your he...,0
4284,6087,hellfire,NaN,The Prophet (peace be upon him) said 'Save you...,0
4285,6088,hellfire,NaN,Hellfire is surrounded by desires so be carefu...,0


In [10]:
# Remove rows with contradictory labels entirely
train = train[~contradictory_labels]

In [11]:
# Duplicate text
train[train.duplicated(subset='text', keep=False)]

,id,keyword,location,text,target
1134,1634,bombing,NaN,Japan on Thursday marks the 70th anniversary o...,1
1156,1665,bombing,Singapore,Japan on Thursday marks the 70th anniversary o...,1
1172,1689,bridge%20collapse,"Mumbai , India",Warne shocked over Australia's epic collapse a...,1
1197,1723,buildings%20burning,"Mackay, QLD, Australia",Mmmmmm I'm burning.... I'm burning buildings I...,1
1199,1725,buildings%20burning,NJ,@themagickidraps not upset with a rally upset ...,1
...,...,...,...,...,...
7600,10855,NaN,NaN,Evacuation order lifted for town of Roosevelt:...,1
7607,10867,NaN,NaN,#stormchase Violent Record Breaking EF-5 El Re...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1


In [12]:
# Remove duplicate (keeping the ones without NaN if possible)
train['null_count'] = train[['keyword', 'location']].isnull().sum(axis=1)
train = train.sort_values('null_count').drop_duplicates(subset='text', keep='first')
train = train.drop(columns='null_count')

train.shape

(7495, 5)

In [13]:
# Missing values
train.isna().sum()

id             0
keyword       56
location    2469
text           0
target         0
dtype: int64

## Analysis by columns

### Target

In [14]:
# Target balance
train.target.value_counts(normalize=True)

target
0    0.573849
1    0.426151
Name: proportion, dtype: float64

### Keyword

In [15]:
# Keyword distribution
train.keyword.value_counts()

keyword
fatalities               45
armageddon               42
deluge                   42
body%20bags              41
damage                   41
                         ..
forest%20fire            19
epicentre                12
threat                   11
inundation               10
radiation%20emergency     9
Name: count, Length: 221, dtype: int64

In [16]:
# Relation NaN and target
pd.crosstab(train.keyword.isna(), train.target, normalize='index')

target,0,1
keyword,,
False,0.575615,0.424385
True,0.339286,0.660714


In [17]:
# Impute NaN with unknown
train['keyword'] = train['keyword'].fillna('unknown')
test['keyword'] = test['keyword'].fillna('unknown')

In [18]:
# Relation top keyword and target
kw = train.groupby('keyword')['target'].agg(['mean', 'count'])
kw = kw[kw['count'] >= 10].sort_values('mean', ascending=False)

print("Top keywords when target = 1")
print(kw.head(10))

print("\nTop keywords when target = 0")
print(kw.sort_values('mean').head(10))

Top keywords when target = 1
                        mean  count
keyword                            
derailment          1.000000     33
debris              1.000000     37
wreckage            1.000000     37
outbreak            0.974359     39
oil%20spill         0.973684     38
typhoon             0.972973     37
suicide%20bombing   0.968750     32
suicide%20bomber    0.967742     31
bombing             0.928571     28
nuclear%20disaster  0.911765     34

Top keywords when target = 0
                 mean  count
keyword                     
aftershock   0.000000     32
body%20bags  0.024390     41
ruin         0.027027     37
blazing      0.029412     34
body%20bag   0.030303     33
electrocute  0.031250     32
screaming    0.055556     36
traumatised  0.057143     35
blew%20up    0.060606     33
panicking    0.060606     33


### Location

In [19]:
# Location distribution
train.location.value_counts()

location
USA                              104
New York                          71
United States                     50
London                            45
Canada                            29
                                ... 
Live On Webcam                     1
Paranaque City                     1
World Wide!!                       1
AFRICA                             1
Est. September 2012 - Bristol      1
Name: count, Length: 3328, dtype: int64

In [20]:
# Relation NaN and target
pd.crosstab(train.location.isna(), train.target, normalize='index')

target,0,1
location,,
False,0.570633,0.429367
True,0.580397,0.419603


In [21]:
# Impute NaN with unknown
train['location'] = train['location'].fillna('unknown')
test['location'] = test['location'].fillna('unknown')

In [22]:
# Relation top location and target
locat = train.groupby('location')['target'].agg(['mean', 'count'])
locat = locat[locat['count'] >= 10].sort_values('mean', ascending=False)

print("Top location when target = 1")
print(locat.head(10))

print("\nTop location when target = 0")
print(locat.sort_values('mean').head(10))

Top location when target = 1
                       mean  count
location                          
India              0.857143     21
Mumbai             0.857143     21
Nigeria            0.739130     23
Earth              0.727273     11
Washington, DC     0.714286     21
Washington, D.C.   0.692308     13
USA                0.644231    104
San Francisco, CA  0.636364     11
Worldwide          0.631579     19
Indonesia          0.615385     13

Top location when target = 0
                     mean  count
location                        
London, England  0.100000     10
ss               0.100000     10
NYC              0.166667     12
Everywhere       0.200000     15
Florida          0.214286     14
New York         0.225352     71
Kenya            0.250000     20
United Kingdom   0.285714     14
Texas            0.300000     10
Los Angeles, CA  0.307692     26


### Text

In [23]:
# Tweet length by class
train['char_len'] = train['text'].str.len()
train['word_len'] = train['text'].str.split().str.len()

print("Characters")
print(train.groupby('target')['char_len'].describe())
print("\nWords")
print(train.groupby('target')['word_len'].describe())

Characters
         count        mean        std   min   25%    50%    75%    max
target                                                                
0       4301.0   95.597303  35.935758   7.0  68.0  101.0  130.0  157.0
1       3194.0  108.031935  29.249195  14.0  88.0  114.0  135.0  151.0

Words
         count       mean       std  min   25%   50%   75%   max
target                                                          
0       4301.0  14.676587  6.159869  1.0  10.0  15.0  19.0  31.0
1       3194.0  15.153100  5.094292  2.0  11.0  15.0  19.0  30.0


In [24]:
# Presence of URLs, mentions and hashtags
train['has_url'] = train['text'].str.contains('http', regex=False)
train['has_mention'] = train['text'].str.contains('@', regex=False)
train['has_hashtag'] = train['text'].str.contains('#', regex=False)

test['has_url'] = test['text'].str.contains('http', regex=False)
test['has_mention'] = test['text'].str.contains('@', regex=False)
test['has_hashtag'] = test['text'].str.contains('#', regex=False)

display(pd.crosstab(train['has_url'], train['target'], normalize='index'))
display(pd.crosstab(train['has_mention'], train['target'], normalize='index'))
display(pd.crosstab(train['has_hashtag'], train['target'], normalize='index'))

target,0,1
has_url,,
False,0.705485,0.294515
True,0.455076,0.544924


target,0,1
has_mention,,
False,0.538279,0.461721
True,0.670129,0.329871


target,0,1
has_hashtag,,
False,0.594047,0.405953
True,0.505828,0.494172


In [25]:
# Check number of languages
train['language'] = train['text'].apply(lambda x: detect(x) if isinstance(x, str) and x.strip() else 'unknown')
train['language'].value_counts()

language
en    7203
de      58
da      25
ca      22
it      19
af      17
fr      17
nl      16
et      14
no      12
id      11
sv      11
tl       9
es       8
ro       8
cy       7
so       7
pt       5
vi       5
pl       4
sw       4
sl       4
hr       2
sq       2
fi       2
tr       1
lt       1
sk       1
Name: count, dtype: int64

# 4. Preprocessing

In [26]:
# Create copy of text
train['text_clean'] = train['text'].str.lower()
test['text_clean'] = test['text'].str.lower()

In [27]:
# Eliminate URLs
train['text_clean'] = train['text_clean'].str.replace(r'http\S+|www\.\S+', '', regex=True)
test['text_clean'] = test['text_clean'].str.replace(r'http\S+|www\.\S+', '', regex=True)

In [28]:
# Remove mentions
train['text_clean'] = train['text_clean'].str.replace(r'@\w+', '', regex=True)
test['text_clean'] = test['text_clean'].str.replace(r'@\w+', '', regex=True)

In [29]:
# Eliminate # symbol
train['text_clean'] = train['text_clean'].str.replace('#', '', regex=False)
test['text_clean'] = test['text_clean'].str.replace('#', '', regex=False)

In [30]:
# Remove HTML tags
train['text_clean'] = train['text_clean'].str.replace(r'<.*?>', ' ', regex=True)
test['text_clean'] = test['text_clean'].str.replace(r'<.*?>', ' ', regex=True)

In [31]:
# Decode HTML entities and fix mojibake
def fix_encoding(text):
    text = html.unescape(text)
    text = re.sub(r'\x89Û.', "'", text)
    text = text.encode('ascii', 'ignore').decode()
    return text

train['text_clean'] = train['text_clean'].apply(fix_encoding)
test['text_clean'] = test['text_clean'].apply(fix_encoding)

In [32]:
# Remove escape characters
train['text_clean'] = train['text_clean'].str.replace(r'[\n\r\t]', ' ', regex=True)
test['text_clean'] = test['text_clean'].str.replace(r'[\n\r\t]', ' ', regex=True)

In [33]:
# Remove punctuation
train['text_clean'] = train['text_clean'].str.replace(f"[{re.escape(string.punctuation)}]", " ", regex=True)
test['text_clean'] = test['text_clean'].str.replace(f"[{re.escape(string.punctuation)}]", " ", regex=True)

In [34]:
# Remove numbers
train['text_clean'] = train['text_clean'].str.replace(r'\d+', ' ', regex=True)
test['text_clean'] = test['text_clean'].str.replace(r'\d+', ' ', regex=True)

In [35]:
# Remove stopwords
stop_words = set(stopwords.words('english'))
stop_words = stop_words - {'no', 'not', 'nor'} 

train['text_clean'] = train['text_clean'].apply(
    lambda x: ' '.join(word for word in x.split() if word not in stop_words)
)

test['text_clean'] = test['text_clean'].apply(
    lambda x: ' '.join(word for word in x.split() if word not in stop_words)
)

In [36]:
# Lemmatization
lemmatizer = WordNetLemmatizer()

train['text_clean'] = train['text_clean'].apply(
    lambda x: ' '.join(lemmatizer.lemmatize(word) for word in x.split())
)

test['text_clean'] = test['text_clean'].apply(
    lambda x: ' '.join(lemmatizer.lemmatize(word) for word in x.split())
)

In [37]:
# Remove extra whitespace
train['text_clean'] = train['text_clean'].str.replace(r'\s+', ' ', regex=True).str.strip()
test['text_clean'] = test['text_clean'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [38]:
# Check empty text
empty_mask = train['text_clean'].str.len() == 0
train[empty_mask]

,id,keyword,location,text,target,char_len,word_len,has_url,has_mention,has_hashtag,language,text_clean
4497,6394,hurricane,NAWF SIDE POKING OUT,@Hurricane_Dame ???????? I don't have them the...,1,56,9,False,True,False,en,
6766,9697,tornado,unknown,@Ayshun_Tornado then don't,0,26,3,False,True,False,en,


In [39]:
# Check empty text
empty_mask_test = test['text_clean'].str.len() == 0
test[empty_mask_test]

,id,keyword,location,text,has_url,has_mention,has_hashtag,text_clean
13,43,unknown,unknown,What if?!,False,False,False,


In [40]:
# Remove rows with empty text_clean (train only)
train = train[train['text_clean'].str.len() > 0]

# 5. Train/validation split

In [41]:
# Separate features from target
X = train.drop(columns='target')
y = train['target']

In [42]:
# Train / validation split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=seed,
    stratify=y
)

# 6. Modeling

## Model 1: Logistic Regression with Bag of Words

In [43]:
# Vectorize + Modeling
pipeline_bow_lr = Pipeline([
    ('bow', CountVectorizer()),
    ('clf', LogisticRegression(random_state=seed, max_iter=1000))
])

pipeline_bow_lr.fit(X_train['text_clean'], y_train)

y_pred_bow_lr = pipeline_bow_lr.predict(X_val['text_clean'])

print("Accuracy:", accuracy_score(y_val, y_pred_bow_lr))
print("F1 score:", f1_score(y_val, y_pred_bow_lr))
print(classification_report(y_val, y_pred_bow_lr))
print(confusion_matrix(y_val, y_pred_bow_lr))

Accuracy: 0.8098732488325551
F1 score: 0.7646573080099092
              precision    recall  f1-score   support

           0       0.81      0.87      0.84       860
           1       0.81      0.72      0.76       639

    accuracy                           0.81      1499
   macro avg       0.81      0.80      0.80      1499
weighted avg       0.81      0.81      0.81      1499

[[751 109]
 [176 463]]


In [44]:
# Kaggle Validation
test_pred1 = pipeline_bow_lr.predict(test['text_clean'])
submission1 = pd.DataFrame({
    'id': test['id'],
    'target': test_pred1
})

submission1.to_csv('01_logistic_regression_bow.csv', index=False)

### Results
| Logistic Regression Baseline with Bag of Words |  | 
| :--- | :--- | 
| **Accuracy** | 0.80987 | 
| **F1 score** | 0.76466 |
| **Kaggle F1 score** | 0.80049 |

## Model 2: Logistic Regression with TF-IDF

In [45]:
# Vectorize + Modeling
pipeline_tfidf_lr = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(random_state=seed, max_iter=1000))
])

pipeline_tfidf_lr.fit(X_train['text_clean'], y_train)

y_pred_tfidf_lr = pipeline_tfidf_lr.predict(X_val['text_clean'])

print("Accuracy:", accuracy_score(y_val, y_pred_tfidf_lr))
print("F1 score:", f1_score(y_val, y_pred_tfidf_lr))
print(classification_report(y_val, y_pred_tfidf_lr))
print(confusion_matrix(y_val, y_pred_tfidf_lr))

Accuracy: 0.799866577718479
F1 score: 0.7427101200686106
              precision    recall  f1-score   support

           0       0.79      0.89      0.84       860
           1       0.82      0.68      0.74       639

    accuracy                           0.80      1499
   macro avg       0.80      0.78      0.79      1499
weighted avg       0.80      0.80      0.80      1499

[[766  94]
 [206 433]]


In [46]:
# Kaggle Validation
test_pred2 = pipeline_tfidf_lr.predict(test['text_clean'])
submission2 = pd.DataFrame({
    'id': test['id'],
    'target': test_pred2
})

submission2.to_csv('02_logistic_regression_tfidf.csv', index=False)

### Results
| Logistic Regression Baseline with TF-IDF |  | 
| :--- | :--- | 
| **Accuracy** | 0.79987 | 
| **F1 score** | 0.74271 |
| **Kaggle F1 score** | 0.79160 |

## Model 3: Naive Bayes with Bag of Words

In [47]:
# Vectorize + Modeling
pipeline_bow_nb = Pipeline([
    ('bow', CountVectorizer()),
    ('clf', MultinomialNB())
])

pipeline_bow_nb.fit(X_train['text_clean'], y_train)

y_pred_bow_nb = pipeline_bow_nb.predict(X_val['text_clean'])

print("Accuracy:", accuracy_score(y_val, y_pred_bow_nb))
print("F1 score:", f1_score(y_val, y_pred_bow_nb))
print(classification_report(y_val, y_pred_bow_nb))
print(confusion_matrix(y_val, y_pred_bow_nb))

Accuracy: 0.8012008005336891
F1 score: 0.7592891760904685
              precision    recall  f1-score   support

           0       0.81      0.85      0.83       860
           1       0.78      0.74      0.76       639

    accuracy                           0.80      1499
   macro avg       0.80      0.79      0.79      1499
weighted avg       0.80      0.80      0.80      1499

[[731 129]
 [169 470]]


In [48]:
# Kaggle Validation
test_pred3 = pipeline_bow_nb.predict(test['text_clean'])
submission3 = pd.DataFrame({
    'id': test['id'],
    'target': test_pred3
})

submission3.to_csv('03_naive_bayes_bow.csv', index=False)

### Results
| Naive Bayes Baseline with Bag of Words |  | 
| :--- | :--- | 
| **Accuracy** | 0.80120 | 
| **F1 score** | 0.75929 |
| **Kaggle F1 score** | 0.79098 |

## Model 4: SVM with Bag of Words

In [49]:
# Vectorize + Modeling
pipeline_bow_svm = Pipeline([
    ('bow', CountVectorizer()),
    ('clf', LinearSVC(random_state=seed, max_iter=5000))
])

pipeline_bow_svm.fit(X_train['text_clean'], y_train)

y_pred_bow_svm = pipeline_bow_svm.predict(X_val['text_clean'])

print("Accuracy:", accuracy_score(y_val, y_pred_bow_svm))
print("F1 score:", f1_score(y_val, y_pred_bow_svm))
print(classification_report(y_val, y_pred_bow_svm))
print(confusion_matrix(y_val, y_pred_bow_svm))

Accuracy: 0.7858572381587725
F1 score: 0.7462450592885376
              precision    recall  f1-score   support

           0       0.81      0.82      0.81       860
           1       0.75      0.74      0.75       639

    accuracy                           0.79      1499
   macro avg       0.78      0.78      0.78      1499
weighted avg       0.79      0.79      0.79      1499

[[706 154]
 [167 472]]


In [50]:
# Kaggle Validation
test_pred4 = pipeline_bow_svm.predict(test['text_clean'])
submission4 = pd.DataFrame({
    'id': test['id'],
    'target': test_pred4
})

submission4.to_csv('04_SVM_bow.csv', index=False)

### Results
| SVM Baseline with Bag of Words |  | 
| :--- | :--- | 
| **Accuracy** | 0.78586 | 
| **F1 score** | 0.74625 |
| **Kaggle F1 score** | 0.77413 |

## Model 5: Logistic Regression with Bag of Words (GridSearchCV)

In [51]:
# Vectorize + Modeling
pipeline = Pipeline([
    ('bow', CountVectorizer()),
    ('clf', LogisticRegression(random_state=seed, max_iter=1000))
])

# GridSearchCV
param_grid = {
    'bow__min_df': [1, 2, 5],
    'bow__max_df': [0.9, 1.0],
    'bow__ngram_range': [(1,1), (1,2)],
    'clf__C': [0.01, 0.1, 1, 10]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    scoring='f1', 
    cv=5, 
    n_jobs=-1  
)

grid_search.fit(X_train['text_clean'], y_train)

print("Best params:", grid_search.best_params_)
print("Best CV F1:", grid_search.best_score_)

Best params: {'bow__max_df': 0.9, 'bow__min_df': 2, 'bow__ngram_range': (1, 1), 'clf__C': 1}
Best CV F1: 0.7499738922542484


In [52]:
# Best modelo validation
best_model = grid_search.best_estimator_

y_pred_best_lr_bow = best_model.predict(X_val['text_clean'])

print("Accuracy:", accuracy_score(y_val, y_pred_best_lr_bow))
print("F1 score:", f1_score(y_val, y_pred_best_lr_bow))
print(classification_report(y_val, y_pred_best_lr_bow))
print(confusion_matrix(y_val, y_pred_best_lr_bow))

Accuracy: 0.8032021347565044
F1 score: 0.757201646090535
              precision    recall  f1-score   support

           0       0.81      0.87      0.83       860
           1       0.80      0.72      0.76       639

    accuracy                           0.80      1499
   macro avg       0.80      0.79      0.80      1499
weighted avg       0.80      0.80      0.80      1499

[[744 116]
 [179 460]]


In [53]:
# Kaggle Validation
test_pred5 = best_model.predict(test['text_clean'])
submission5 = pd.DataFrame({
    'id': test['id'],
    'target': test_pred5
})

submission5.to_csv('05_best_logistic_regression_bow.csv', index=False)

### Results
| Logistic Regression Baseline with Bag of Words |  | 
| :--- | :--- | 
| **Accuracy** | 0.80320 | 
| **F1 score** | 0.75720 |
| **Kaggle F1 score** | 0.79436 |

## Model 6: Logistic Regression with Bag of Words (More features)

In [54]:
# Preprocessing + Modeling
preprocessor = ColumnTransformer(
    transformers=[
        ('text', CountVectorizer(), 'text_clean'),
        ('keyword', OneHotEncoder(handle_unknown='ignore'), ['keyword']),
        ('bools', 'passthrough', ['has_url', 'has_mention', 'has_hashtag'])
    ]
)

lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(random_state=seed, max_iter=1000))
])

lr_pipeline.fit(X_train, y_train)  

y_pred_lr = lr_pipeline.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred_lr))
print("F1 score:", f1_score(y_val, y_pred_lr))
print(classification_report(y_val, y_pred_lr))
print(confusion_matrix(y_val, y_pred_lr))

Accuracy: 0.8072048032021347
F1 score: 0.7629204265791633
              precision    recall  f1-score   support

           0       0.81      0.87      0.84       860
           1       0.80      0.73      0.76       639

    accuracy                           0.81      1499
   macro avg       0.81      0.80      0.80      1499
weighted avg       0.81      0.81      0.81      1499

[[745 115]
 [174 465]]


In [55]:
# Kaggle Validation
test_pred6 = best_model.predict(test['text_clean'])
submission6 = pd.DataFrame({
    'id': test['id'],
    'target': test_pred6
})

submission6.to_csv('06_logistic_regression_more_features.csv', index=False)

### Results
| Logistic Regression Baseline with Bag of Words (More Features) |  | 
| :--- | :--- | 
| **Accuracy** | 0.80720 | 
| **F1 score** | 0.76292 |
| **Kaggle F1 score** | 0.79436 |

## Model 7: DistilBERT

In [56]:
# Train / validation split
X = train.drop(columns='target')
y = train['target']

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=seed,
    stratify=y
)

In [57]:
# Convert Dataset for model
train_dataset = Dataset.from_dict({
    "text": X_train["text"].tolist(),
    "label": y_train.tolist()
})

val_dataset = Dataset.from_dict({
    "text": X_val["text"].tolist(),
    "label": y_val.tolist()
})

test_dataset = Dataset.from_dict({
    "text": test["text"].tolist()
})

In [58]:
# Tokenize
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.remove_columns("text")
val_dataset = val_dataset.remove_columns("text")
test_dataset = test_dataset.remove_columns("text")

Map: 100%|██████████| 3263/3263 [00:00<00:00, 24559.04 examples/s]


In [59]:
# Cast to PyTorch tensors 
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

In [60]:
# Modeling
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="./logs",
    logging_steps=50,
    seed=seed
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9460.91it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [61]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.444219,0.390260,0.842562,0.800338
2,0.296472,0.412732,0.843896,0.812500
3,0.217331,0.430798,0.843896,0.809446


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.53it/s]


TrainOutput(global_step=1125, training_loss=0.33940289688110353, metrics={'train_runtime': 85.8557, 'train_samples_per_second': 209.444, 'train_steps_per_second': 13.103, 'total_flos': 595507190658048.0, 'train_loss': 0.33940289688110353, 'epoch': 3.0})

In [62]:
# Kaggle Validation
test_pred7 = trainer.predict(test_dataset)
y_test_pred7 = np.argmax(test_pred7.predictions, axis=1)

submission7 = pd.DataFrame({
    'id': test['id'],
    'target': y_test_pred7
})

submission7.to_csv('07_distilbert.csv', index=False)

### Results
| DistilBERT |  | 
| :--- | :--- | 
| **Accuracy** | 0.84390 | 
| **F1 score** | 0.81250 |
| **Kaggle F1 score** | 0.83328 |

## Model 8: RoBERTa

In [63]:
# Convert Dataset for model
train_dataset = Dataset.from_dict({
    "text": X_train["text"].tolist(),
    "label": y_train.tolist()
})

val_dataset = Dataset.from_dict({
    "text": X_val["text"].tolist(),
    "label": y_val.tolist()
})

test_dataset = Dataset.from_dict({
    "text": test["text"].tolist()
})

In [64]:
# Tokenize
tokenizer = RobertaTokenizerFast.from_pretrained(
    "roberta-base"
)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.remove_columns("text")
val_dataset = val_dataset.remove_columns("text")
test_dataset = test_dataset.remove_columns("text")

Map: 100%|██████████| 3263/3263 [00:00<00:00, 26575.87 examples/s]


In [65]:
# Cast to PyTorch tensors 
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

In [66]:
# Modeling
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate = 1e-5,
    num_train_epochs = 5,    
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="./logs",
    logging_steps=50,
    seed=seed
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 7951.33it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [67]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.606768,0.658321,0.642428,0.624650
2,0.575590,0.641344,0.683789,0.631415
3,0.543706,0.611209,0.702468,0.648819
4,0.536311,0.593084,0.711141,0.638866
5,0.486878,0.599403,0.719146,0.660757


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.08it/s]


TrainOutput(global_step=1875, training_loss=0.5659154856363933, metrics={'train_runtime': 282.9203, 'train_samples_per_second': 105.931, 'train_steps_per_second': 6.627, 'total_flos': 1971359582284800.0, 'train_loss': 0.5659154856363933, 'epoch': 5.0})

### Results
| DistilBERT |  | 
| :--- | :--- | 
| **Accuracy** | 0.719146 | 
| **F1 score** | 0.660757 |
| **Kaggle F1 score** | - |